# Querying the lakehouse from Python

Connects to Trino on port 8080. Works from VS Code (local `.venv`) or from the Jupyter container at http://localhost:8888.

In [ ]:
import os

import pandas as pd
import trino

conn = trino.dbapi.connect(host=os.getenv("TRINO_HOST", "localhost"), port=8080, user="amanda", catalog="iceberg", schema="raw")


def sql(query: str) -> pd.DataFrame:
    cur = conn.cursor()
    cur.execute(query)
    return pd.DataFrame(cur.fetchall(), columns=[c[0] for c in cur.description])

In [ ]:
sql("SHOW TABLES")

## Washington St near Hyde Park Ave, summer counts

In [ ]:
sql("""
SELECT year,
       round(avg(CAST(day_bikes AS integer))) AS avg_bikes,
       round(avg(CAST(day_veh AS integer))) AS avg_vehicles
FROM boston_bike_counts_daily
WHERE location_name = 'Washington Street north of Bexley Road' AND season = 'Summer'
GROUP BY year
ORDER BY year
""")

## NYC protected bike lane segments by install year

In [ ]:
sql("""
SELECT year(CAST(substr(instdate, 1, 10) AS date)) AS install_year, count(*) AS protected_segments
FROM nyc_bike_routes
WHERE 'Protected' IN (ft_facilit, tf_facilit) AND instdate IS NOT NULL
GROUP BY 1
ORDER BY 1 DESC
LIMIT 10
""")

## Live stations (Postgres) next to a year of trips (Iceberg)

Run `uv run bikelake stations` or materialize `station_status` in Dagster first.

In [ ]:
sql(open("../sql/02_live_and_history.sql").read().split(";")[0])

## Snapshots and time travel

Every load is a snapshot. Read the table as it was at the first load.

In [ ]:
snapshots = sql('SELECT snapshot_id, operation, committed_at FROM "boston_bike_counts_daily$snapshots" ORDER BY committed_at')
snapshots

In [ ]:
first = snapshots.snapshot_id.iloc[0]
sql(f"SELECT count(*) AS rows_at_first_load FROM boston_bike_counts_daily FOR VERSION AS OF {first}")